<a href="https://colab.research.google.com/github/OvFabri/App--TareaGo/blob/main/Actividad4_MineriadeDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar el CSV original
df = pd.read_csv("online_retail_II.csv")

# Homologar los nombres de las columnas con los que usaremos en el notebook.
# El CSV tiene "Customer ID", "Invoice" y "Price".
df = df.rename(columns={
    "Customer ID": "CustomerID",
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice"
})

# Preparación básica
df = df.dropna(subset=["CustomerID", "Description"]).copy()
df["CustomerID"] = df["CustomerID"].astype("int64")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

# Excluir cancelaciones/devoluciones y precios no válidos
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()

# Fecha de corte: solo el período anterior crea las variables predictoras
cutoff_date = pd.Timestamp("2011-06-01")

df_hist = df[df["InvoiceDate"] < cutoff_date].copy()
df_futuro = df[df["InvoiceDate"] >= cutoff_date].copy()

# Importe de cada renglón, calculado con el período histórico
df_hist["Total_Line"] = df_hist["Quantity"] * df_hist["UnitPrice"]

# Variables históricas agregadas por cliente
features = (
    df_hist.groupby("CustomerID")
    .agg(
        Num_Invoices=("InvoiceNo", "nunique"),
        Total_Spent=("Total_Line", "sum")
    )
)

# Tamaño promedio de la canasta:
# primero suma las unidades de cada factura y luego promedia por cliente
basket_size = (
    df_hist.groupby(["CustomerID", "InvoiceNo"])["Quantity"]
    .sum()
    .groupby("CustomerID")
    .mean()
    .rename("Avg_Basket_Size")
)

features = features.join(basket_size)

# Target: 1 si el cliente vuelve a comprar después del corte; 0 si no
customers_future = set(df_futuro["CustomerID"].unique())
features["Will_Return"] = features.index.isin(customers_future).astype(int)

# Matriz de variables y variable objetivo
columnas_x = ["Num_Invoices", "Total_Spent", "Avg_Basket_Size"]
X = features[columnas_x]
y = features["Will_Return"]

print("Distribución del target (0 = no regresa, 1 = regresa):")
print(y.value_counts(normalize=True).sort_index())

# Partición 80/20, reproducible y estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Escalar: ajustar solo con Train y aplicar esa transformación a Test
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=columnas_x,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=columnas_x,
    index=X_test.index
)

print("\nPreparación terminada.")
print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Distribución del target (0 = no regresa, 1 = regresa):
Will_Return
0    0.472532
1    0.527468
Name: proportion, dtype: float64

Preparación terminada.
X_train: (3946, 3)
X_test: (987, 3)
y_train: (3946,)
y_test: (987,)


In [2]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    make_scorer,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Recall de la clase 0:
# Will_Return=0 representa clientes que no regresan (churn).
recall_churn = make_scorer(
    recall_score,
    pos_label=0,
    zero_division=0
)

# Validación cruzada estratificada de cinco particiones
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# solver="saga" permite comparar penalizaciones L1 y L2
modelo_base = LogisticRegression(
    solver="saga",
    max_iter=5000,
    class_weight="balanced",
    random_state=42
)

# Valores de regularización e hipótesis de penalización por comparar
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"]
}

busqueda = GridSearchCV(
    estimator=modelo_base,
    param_grid=param_grid,
    scoring=recall_churn,
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

# Ajustar exclusivamente con el conjunto Train
busqueda.fit(X_train_scaled, y_train)

modelo_logistico = busqueda.best_estimator_

# Tabla de mejores hiperparámetros
tabla_mejores_hiperparametros = pd.DataFrame([{
    "C": busqueda.best_params_["C"],
    "penalty": busqueda.best_params_["penalty"],
    "Recall churn promedio CV": busqueda.best_score_,
    "Métrica de selección": "Recall de la clase 0"
}])

print("Mejores hiperparámetros:")
display(tabla_mejores_hiperparametros)

# Evaluación en Test para compartir con Persona 4 y Persona 5
y_pred = modelo_logistico.predict(X_test_scaled)
y_proba = modelo_logistico.predict_proba(X_test_scaled)[:, 1]

print("Matriz de confusión (filas = real, columnas = predicción; clases [0, 1]):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nReporte de clasificación en Test:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["No regresa (churn)", "Regresa"],
    zero_division=0
))

print("ROC-AUC en Test:", round(roc_auc_score(y_test, y_proba), 4))

Mejores hiperparámetros:


,C,penalty,Recall churn promedio CV,Métrica de selección
0,0.01,l1,0.877212,Recall de la clase 0


Matriz de confusión (filas = real, columnas = predicción; clases [0, 1]):
[[414  52]
 [263 258]]

Reporte de clasificación en Test:
                    precision    recall  f1-score   support

No regresa (churn)       0.61      0.89      0.72       466
           Regresa       0.83      0.50      0.62       521

          accuracy                           0.68       987
         macro avg       0.72      0.69      0.67       987
      weighted avg       0.73      0.68      0.67       987

ROC-AUC en Test: 0.7457
